In [0]:
-- League - Most Traveled Players
-- Players with the most teams and stints (from dim_player_team).
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2 AS min_teams,  -- minimum different teams
    25 AS top_n      -- rows to show
)
SELECT * FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY COUNT(DISTINCT pt.team_id) DESC, COUNT(*) DESC) AS rank,
         p.full_name,
         COUNT(*)                   AS stints,
         COUNT(DISTINCT pt.team_id) AS distinct_teams,
         MIN(pt.valid_from)         AS first_seen,
         MAX(pt.valid_to)           AS last_seen,
         p.player_id
  FROM nhl.gold.dim_player_team pt
  JOIN nhl.gold.dim_player p ON pt.player_id = p.player_id
  GROUP BY p.full_name, p.player_id
  HAVING COUNT(DISTINCT pt.team_id) >= (SELECT min_teams FROM params)
) ranked
WHERE rank <= (SELECT top_n FROM params)
ORDER BY rank;